# CS-781 Phase 6 · Abstractive Summarization · LegalEncDec

This is a **complete Phase-6 workflow**, not the earlier Section-1 scaffold. Edit the checkpoint
and tokenizer paths, select one preset, then Run All. It loads your trained Phase-1 backbone,
fine-tunes on Phase-6 training data, validates, restores the best model, generates every test
summary, and writes `prediction.json` at the root of `submission_<EXP>.zip`.

**What changed from Phase 5:** the Phase-1 loading pattern and the supplied tokenizer/model
definitions are preserved. Statute classification heads, pooling, BCE and thresholds are
replaced with autoregressive target-token losses and cached text generation. Encoder-decoder
keeps **all encoder layers, decoder layers, cross-attention and the tied language-model head**.
Decoder-only and Mamba use `Document: <text> Summary:` after Phase-1 whitespace normalization.

**Exact source boundary:** tokenizer and backbone cells are byte-for-byte copies from the
corresponding supplied Phase-5 notebook. Task wrappers and PEFT code are new and clearly
separate. File extension/name (`mamba.pt`, `best.pt`, etc.) is immaterial; its contents must use
the Phase-1 `model`/`cfg` format. A bare state_dict lacks the config and is rejected.

**Budget:** 90 minutes is a configurable estimate, not a guarantee. A synchronized benchmark
measures backward/optimizer work and max-length generation. Auto-fit reserves validation,
all test rows, teacher SFT for KD, I/O and a safety margin. If fixed costs exceed budget,
the run stops with an actionable error; it never silently submits fewer test rows.
Each notebook is for its own single GPU session. Running these files does not launch sessions.

**Validation limits:** synthetic CPU tests are recorded below; no real checkpoints, actual
Phase-6 data, Kaggle T4 FP16 runs, official BERTScore or quality results have been measured here.
Hyperparameters are starting assumptions. Production performance is not certified.


## 1. Configuration
Set `CKPT_PATH` and `TOK_PATH` to your existing Phase-1 assets. Then choose `EXPERIMENT`.


In [25]:
import os, sys, json, re, math, time, random, zipfile, heapq, hashlib, copy, warnings
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass
from contextlib import nullcontext
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint

KIND = 'encdec'                        # Fixed: use the other notebook for another backbone.
EXPERIMENT = 'E2_prefix'               # Change only this for another technique.
CKPT_PATH = "/kaggle/input/models/karankumar9939/cs-781-project-encoder-decoder-model/pytorch/default/1/encoder_decoder_checpoint/best.pt"  # YOUR PHASE-1 checkpoint, not Phase 5.
TOK_PATH = "/kaggle/input/models/karankumar9939/cs-781-project-encoder-decoder-model/pytorch/default/1/encoder_decoder_checpoint/bpe_merges.json"
# For Mamba, use best.pt instead if that is your actual Phase-1 filename.
# Required checkpoint contents: {"model": state_dict, "cfg": config_dict, "step": int}.
EXPECTED_TOKENIZER_SHA256 = ""        # Optional known Phase-1 digest; always printed/recorded.
LOCAL_TRAIN_JSON = ""                 # Set both local paths to run with Kaggle Internet off.
LOCAL_TEST_JSON = ""
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_REVISION = "main"                  # Pin a dataset commit for reproducible final experiments.
OUTPUT_DIR = "/kaggle/working/phase6_summ_encdec"
EOS_PLACEHOLDER = "<|endoftext|>"
SEED = 42
PRESETS = {'E1_houlsby': {'method': 'houlsby', 'lr': 0.0002}, 'E2_prefix': {'method': 'prefix', 'lr': 0.001}, 'E3_kd': {'method': 'kd', 'lr': 3e-05}, 'E4_qlora': {'method': 'qlora', 'lr': 0.0002}, 'E5_full': {'method': 'full', 'lr': 3e-05}, 'E6_lorma': {'method': 'lorma', 'lr': 0.0002}}
EXP = PRESETS[EXPERIMENT]

# Long-document selection: windows together fit MAX_SOURCE_TOKENS.
MAX_SOURCE_TOKENS = 960
CONTEXT_POLICY = "uniform"            # uniform | head_tail
SELECTION_WINDOWS = 4
MAX_NEW_TOKENS, MIN_NEW_TOKENS = 256, 64
DO_SAMPLE, TEMPERATURE, TOP_P, TOP_K = False, .8, .9, 50
REPETITION_PENALTY, NO_REPEAT_NGRAM = 1.10, 3
DECODER_START_ID = None               # Encdec: None uses EOS; Phase-1 start-token convention UNVERIFIED.

# Single T4: float32 parameters, float16 autocast, GradScaler; no BF16.
EPOCHS, GRAD_ACCUM = 3, 8             # One unpadded document per microbatch, effective batch 8.
GRAD_CKPT = False                     # Mask-safe wrapper for causal models; native path for unpadded encdec.
RANK, ALPHA, ADAPTER_DIM, PROMPT_TOKENS = 8, 16, 64, 32
WEIGHT_DECAY, CLIP, WARMUP_FRACTION = .01, 1., .06
DEV_FRACTION, MAX_DEV_DOCS = .10, 32  # Fixed deterministic dev subset for expensive generation.
TRAIN_DOCS_PER_EPOCH = None          # Optional cap; measured budget may set a lower cap.
TOTAL_MINUTES, SAFETY_FRACTION = 90., .20
BENCHMARK_MARGIN = 1.25
CHECKPOINT_RESERVE_SECONDS = 60.
KD_ALPHA, KD_TEMPERATURE = .5, 2.
TEACHER_LR, TEACHER_BUDGET_FRACTION, TEACHER_MAX_UPDATES = 3e-5, .35, 1000
RUN_PIPELINE = True                  # Run All loads Phase 1 and executes training through ZIP.

assert 0 <= MIN_NEW_TOKENS < MAX_NEW_TOKENS
assert TEMPERATURE>0 and 0<TOP_P<=1 and TOP_K>=0 and REPETITION_PENALTY>=1
assert 0<DEV_FRACTION<1 and GRAD_ACCUM>=1 and EPOCHS>=1
assert 0<=SAFETY_FRACTION<1 and TOTAL_MINUTES>0
assert MAX_DEV_DOCS is None or MAX_DEV_DOCS>0
assert TRAIN_DOCS_PER_EPOCH is None or TRAIN_DOCS_PER_EPOCH>0
assert 0<=KD_ALPHA<=1 and KD_TEMPERATURE>0 and 0<TEACHER_BUDGET_FRACTION<1
assert SELECTION_WINDOWS>=1 and MAX_SOURCE_TOKENS>0
RUN_TAG = time.strftime("%Y%m%d-%H%M%S")  # Preserve earlier runs and avoid stale submission files.
EXP_DIR=Path(OUTPUT_DIR)/EXPERIMENT/RUN_TAG; EXP_DIR.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEVICE.type=="cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
print("Device:",DEVICE,"| GPU:",torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else "CPU",
      "| PyTorch:",torch.__version__,"| experiment:",EXPERIMENT)
if DEVICE.type!="cuda": print("CPU mode is for small correctness tests. Select a GPU for the real run.")


Device: cuda:0 | GPU: Tesla T4 | PyTorch: 2.11.0+cu130 | experiment: E2_prefix


## 2. Exact Phase-1 tokenizer and backbone definitions from the Phase-5 reference
No changes inside the next two cells. Full model construction remains checkpoint-compatible.


In [26]:
CHAR_MAP = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-",
    "\u2212": "-", "\u2026": "...", "\u00a0": " ", "\u200b": "", "\ufeff": "", "\u00ad": "",
})
CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

def normalize_text(t: str, keep_newlines: bool) -> str:
    """Character and whitespace normalisation. Applied to train text AND test prompts."""
    t = CTRL.sub(" ", t.translate(CHAR_MAP))
    if keep_newlines:
        t = re.sub(r"[ \t]+", " ", t)
        t = re.sub(r" ?\n ?", "\n", t)
        t = re.sub(r"\n{3,}", "\n\n", t)
    else:
        t = re.sub(r"\s+", " ", t)
    return t.strip()

PRETOKEN_RE = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+""")

class ByteBPE:
    def __init__(self, merges=None):
        self.merges = [tuple(m) for m in (merges or [])]
        self.rank = {p: i for i, p in enumerate(self.merges)}
        self.vocab = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            self.vocab.append(self.vocab[a] + self.vocab[b])
        self.eos_id = len(self.vocab)
        self.vocab.append(EOS_PLACEHOLDER.encode())
        self.vocab_size = len(self.vocab)
        self._cache = {}

    # ---------- training ----------
    @staticmethod
    def train(chunk_counts, vocab_size, min_freq=3, log_every=2000):
        n_merges = vocab_size - 256 - 1                      # last id is EOS
        items = [(c.encode("utf-8"), f) for c, f in chunk_counts.items() if f >= min_freq]
        words = [list(b) for b, _ in items]
        freqs = [f for _, f in items]
        pair_cnt, where = defaultdict(int), defaultdict(set)
        for i, w in enumerate(words):
            for p in zip(w, w[1:]):
                pair_cnt[p] += freqs[i]; where[p].add(i)
        heap = [(-c, p) for p, c in pair_cnt.items()]
        heapq.heapify(heap)
        merges, t0 = [], time.time()
        while len(merges) < n_merges:
            best = None
            while heap:                                       # lazy deletion
                negc, p = heapq.heappop(heap)
                c = pair_cnt.get(p, 0)
                if c <= 0: continue
                if c != -negc:
                    heapq.heappush(heap, (-c, p)); continue
                best = p; break
            if best is None: break
            a, b = best; new_id = 256 + len(merges); merges.append(best)
            changed = set()
            for i in list(where[best]):
                w, f = words[i], freqs[i]
                out, j, n, hit = [], 0, len(w), False
                while j < n:
                    if j < n - 1 and w[j] == a and w[j + 1] == b:
                        out.append(new_id); j += 2; hit = True
                    else:
                        out.append(w[j]); j += 1
                if not hit: continue
                for q in zip(w, w[1:]): pair_cnt[q] -= f
                for q in zip(out, out[1:]):
                    pair_cnt[q] += f; where[q].add(i); changed.add(q)
                words[i] = out
            where.pop(best, None); pair_cnt.pop(best, None)
            for q in changed:
                if pair_cnt[q] > 0: heapq.heappush(heap, (-pair_cnt[q], q))
            if len(merges) % log_every == 0:
                print(f"  merges {len(merges):6d}/{n_merges} | {time.time() - t0:5.0f}s")
        return ByteBPE(merges)

    # ---------- encoding / decoding ----------
    def _encode_chunk(self, chunk):
        hit = self._cache.get(chunk)
        if hit is not None: return hit
        ids = list(chunk.encode("utf-8"))
        while len(ids) > 1:
            best_r, best_p = None, None
            for p in zip(ids, ids[1:]):
                r = self.rank.get(p)
                if r is not None and (best_r is None or r < best_r):
                    best_r, best_p = r, p
            if best_p is None: break
            a, b = best_p; new_id = 256 + best_r; out, j = [], 0
            while j < len(ids):
                if j < len(ids) - 1 and ids[j] == a and ids[j + 1] == b:
                    out.append(new_id); j += 2
                else:
                    out.append(ids[j]); j += 1
            ids = out
        if len(self._cache) > 3_000_000: self._cache.clear()
        self._cache[chunk] = ids
        return ids

    def encode(self, text):
        ids = []
        for chunk in PRETOKEN_RE.findall(text):
            ids.extend(self._encode_chunk(chunk))
        return ids

    def decode(self, ids):
        data = b"".join(self.vocab[i] for i in ids if i != self.eos_id)
        return data.decode("utf-8", errors="replace")

    def save(self, path):
        json.dump({"merges": self.merges, "eos": EOS_PLACEHOLDER}, open(path, "w"))

    @staticmethod
    def load(path):
        return ByteBPE(json.load(open(path))["merges"])

In [27]:
@dataclass
class EncDecConfig:
    vocab_size: int
    n_ctx: int = 1280
    d_model: int = 768
    n_head: int = 12
    n_enc: int = 6
    n_dec: int = 6
    n_ffn: int = 2048
    rope_base: float = 10000.0

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.weight

def build_rope(head_dim, n_ctx, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(n_ctx).float(), inv)
    return freqs.cos(), freqs.sin()                          # (n_ctx, head_dim/2)

def apply_rope(x, cos, sin):                                 # x: (B, H, T, hd); cos/sin: (1, 1, T, hd/2)
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)

class Attention(nn.Module):
    """Multi-head attention used for encoder self-, decoder self- and cross-attention."""
    def __init__(self, c: EncDecConfig, rope: bool):
        super().__init__()
        self.nh, self.hd, self.rope = c.n_head, c.d_model // c.n_head, rope
        self.q = nn.Linear(c.d_model, c.d_model, bias=False)
        self.k = nn.Linear(c.d_model, c.d_model, bias=False)
        self.v = nn.Linear(c.d_model, c.d_model, bias=False)
        self.o = nn.Linear(c.d_model, c.d_model, bias=False)
    def _split(self, t):
        B, T, _ = t.shape
        return t.view(B, T, self.nh, self.hd).transpose(1, 2)
    def project_kv(self, ctx):
        return self._split(self.k(ctx)), self._split(self.v(ctx))
    def forward(self, x, ctx=None, cos=None, sin=None, mask=None, causal=False, cache=None, kv=None):
        B, T, _ = x.shape
        q = self._split(self.q(x))
        new_cache = None
        if kv is not None:                                   # cross-attention with precomputed K/V
            k, v = kv
        else:
            k, v = self.project_kv(x if ctx is None else ctx)
            if self.rope:
                q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
            if cache is not None:                            # generation: grow the decoder KV-cache
                if cache[0] is not None:
                    k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
                new_cache = (k, v)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                           is_causal=(causal and mask is None and cache is None))
        return self.o(y.transpose(1, 2).reshape(B, T, -1)), new_cache

class SwiGLU(nn.Module):
    def __init__(self, c: EncDecConfig):
        super().__init__()
        self.gate = nn.Linear(c.d_model, c.n_ffn, bias=False)
        self.up = nn.Linear(c.d_model, c.n_ffn, bias=False)
        self.down = nn.Linear(c.n_ffn, c.d_model, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class EncBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n1, self.attn = RMSNorm(c.d_model), Attention(c, rope=True)
        self.n2, self.mlp = RMSNorm(c.d_model), SwiGLU(c)
    def forward(self, x, cos, sin, mask=None):
        h, _ = self.attn(self.n1(x), cos=cos, sin=sin, mask=mask)          # bidirectional
        x = x + h
        return x + self.mlp(self.n2(x))

class DecBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n1, self.self_attn = RMSNorm(c.d_model), Attention(c, rope=True)
        self.n2, self.cross = RMSNorm(c.d_model), Attention(c, rope=False)
        self.n3, self.mlp = RMSNorm(c.d_model), SwiGLU(c)
    def forward(self, x, enc_out, enc_mask, cos, sin, cache=None, cross_kv=None):
        h, cache = self.self_attn(self.n1(x), cos=cos, sin=sin, causal=True, cache=cache)
        x = x + h
        h, _ = self.cross(self.n2(x), ctx=enc_out, mask=enc_mask, kv=cross_kv)
        x = x + h
        return x + self.mlp(self.n3(x)), cache

class LegalEncDec(nn.Module):
    def __init__(self, c: EncDecConfig):
        super().__init__()
        self.c = c
        self.emb = nn.Embedding(c.vocab_size, c.d_model)               # shared by encoder, decoder, head
        self.enc_blocks = nn.ModuleList([EncBlock(c) for _ in range(c.n_enc)])
        self.enc_norm = RMSNorm(c.d_model)
        self.dec_blocks = nn.ModuleList([DecBlock(c) for _ in range(c.n_dec)])
        self.dec_norm = RMSNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.emb.weight                               # weight tying
        cos, sin = build_rope(c.d_model // c.n_head, c.n_ctx, c.rope_base)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.grad_ckpt = False
        self.apply(self._init)
        for n, p in self.named_parameters():                             # scaled residual init
            if n.endswith(".o.weight") or n.endswith("mlp.down.weight"):
                n_res = 2 * c.n_enc if n.startswith("enc_blocks") else 3 * c.n_dec
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(n_res))
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def encode(self, ids, enc_mask=None):
        T = ids.size(1)
        x = self.emb(ids)
        cos, sin = self.cos[:T][None, None], self.sin[:T][None, None]
        for blk in self.enc_blocks:
            if self.grad_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(blk, x, cos, sin, enc_mask, use_reentrant=False)
            else:
                x = blk(x, cos, sin, enc_mask)
        return self.enc_norm(x)

    def decode(self, dec_ids, enc_out, enc_mask=None):                   # teacher-forced, full sequence
        T = dec_ids.size(1)
        x = self.emb(dec_ids)
        cos, sin = self.cos[:T][None, None], self.sin[:T][None, None]
        for blk in self.dec_blocks:
            if self.grad_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(
                    lambda x_, e_, b=blk: b(x_, e_, enc_mask, cos, sin)[0], x, enc_out, use_reentrant=False)
            else:
                x, _ = blk(x, enc_out, enc_mask, cos, sin)
        return self.head(self.dec_norm(x))

    def forward(self, enc_ids, dec_ids, targets=None, enc_mask=None):
        enc_out = self.encode(enc_ids, enc_mask)
        logits = self.decode(dec_ids, enc_out, enc_mask)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        return logits, loss

    # ---------- incremental generation ----------
    def cross_kv(self, enc_out):
        return [blk.cross.project_kv(enc_out) for blk in self.dec_blocks]
    def decode_step(self, tok, pos, caches, cross, enc_mask):
        x = self.emb(tok)                                                # tok: (B, 1)
        cos, sin = self.cos[pos:pos + 1][None, None], self.sin[pos:pos + 1][None, None]
        new = []
        for i, blk in enumerate(self.dec_blocks):
            x, c_new = blk(x, None, enc_mask, cos, sin, cache=caches[i], cross_kv=cross[i])
            new.append(c_new)
        return self.head(self.dec_norm(x)), new

In [28]:
CONFIG_CLASS, MODEL_CLASS = EncDecConfig, LegalEncDec


## 3. Load the Phase-1 checkpoint — this is where mamba.pt / best.pt is used

`main()` always calls `load_assets()`, then `build_backbone()`. The latter constructs the
architecture from `ck["cfg"]` and calls `load_state_dict(ck["model"], strict=True)` **before**
adapting it. There is no disabled-load flag or random-model fallback. Both full teacher and
student start from those Phase-1 tensors. Your own trusted pickle checkpoint is loaded with
`weights_only=False`, matching Phase 5. File paths are placeholders you must set in Section 1.


In [29]:
def build_backbone():
    # Same Phase-1 contract as Phase 5. No Phase-5 classifier checkpoint is used.
    model = MODEL_CLASS(p1_cfg)
    model.load_state_dict(ck["model"], strict=True)
    model.grad_ckpt = False  # Wrapper below handles checkpointing without dropping masks.
    return model

def load_assets():
    global ck, p1_cfg, tokenizer, EOS_ID, BOS_ID, TOKENIZER_SHA256
    for path in (CKPT_PATH, TOK_PATH):
        if not Path(path).is_file():
            raise FileNotFoundError(f"Missing Phase-1 asset: {path}. Set CKPT_PATH/TOK_PATH in Section 1.")
    # Only load your own trusted Phase-1 checkpoint, as in the reference notebooks.
    ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
    if not isinstance(ck, dict) or not {"cfg", "model"} <= ck.keys():
        raise ValueError("Expected Phase-1 {'cfg': config_dict, 'model': state_dict, 'step': int}.")
    p1_cfg = CONFIG_CLASS(**ck["cfg"])
    tokenizer = ByteBPE.load(TOK_PATH)
    EOS_ID = tokenizer.eos_id
    BOS_ID = EOS_ID if DECODER_START_ID is None else DECODER_START_ID
    gap = p1_cfg.vocab_size - tokenizer.vocab_size
    if (KIND != "encdec" and gap != 0) or (KIND == "encdec" and not 0 <= gap <= 256):
        raise ValueError("Tokenizer/checkpoint vocabulary mismatch; use the original Phase-1 merge file.")
    if not 0 <= BOS_ID < p1_cfg.vocab_size:
        raise ValueError("Invalid decoder-start token.")
    TOKENIZER_SHA256 = hashlib.sha256(Path(TOK_PATH).read_bytes()).hexdigest()
    if EXPECTED_TOKENIZER_SHA256 and TOKENIZER_SHA256 != EXPECTED_TOKENIZER_SHA256:
        raise ValueError("Wrong tokenizer SHA-256.")
    print("PHASE-1 CHECKPOINT:", CKPT_PATH, "| step:", ck.get("step"))
    print("Checkpoint config:", ck["cfg"])
    print("Tokenizer:", TOK_PATH, "| SHA-256:", TOKENIZER_SHA256)
    if not EXPECTED_TOKENIZER_SHA256:
        print("NOTE: hash recorded, but tokenizer pairing is not independently authenticated.")
    if KIND == "encdec" and DECODER_START_ID is None:
        print("ASSUMPTION: EOS is the decoder-start token; set DECODER_START_ID if Phase 1 used another ID.")



## 4. Adaptations, full teacher and half-depth student

LORMA implements the requested `W'=(I+(alpha/r)BA)W0` on the output side, with zero B, no
dropout, and unchanged frozen bias. This matches the specified formula, not independently
verified reference [12]. LoRA is additive. The tied embedding/head stays intact and frozen
for PEFT. Linear updates include attention, FFN and (for Mamba) input/output projections.

Hybrid post-block bottlenecks are a **Pfeiffer-style extension**; canonical Transformer adapter
placement does not map uniquely to Mamba. Encoder-decoder Houlsby inserts adapters after
attention residuals (including decoder cross-attention) and FFN residuals. Prefix tuning
trains encoder self-attention K/V prefixes only; the full decoder is retained but frozen.
Prompt tuning prepends trainable embeddings to causal inputs once, including cache prefill.

The QLoRA preset uses **actual packed NF4 + LoRA**, implemented here from scratch, with
block scales and two 4-bit codes per byte. It uses regular AdamW, without double quantization,
paged optimizer or fused dequantization kernels: it is a reduced QLoRA implementation.
Dense dequantization temporaries can erase peak-memory savings and slow generation.
NF4 table source: https://github.com/bitsandbytes-foundation/bitsandbytes/blob/main/bitsandbytes/functional.py

KD retains odd-indexed layers after strict Phase-1 loading: 6/12 decoder layers, 3+3/6+6
encoder-decoder layers, or 10/20 hybrid layers. The hybrid becomes `MA` repeated five times
(5 Mamba + 5 attention), not the original 3:1 mixture. This choice is explicit in saved
metadata. Shared embeddings mean half-depth is not half the parameter count; size is checked.


In [30]:
class LowRankLinear(nn.Module):
    """LoRA, or exact left-multiplicative LORMA: W'=(I+sBA)W0."""
    def __init__(self, base, method, rank=8, alpha=16):
        super().__init__()
        self.base, self.method, self.scale = base, method, alpha / rank
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(rank, base.out_features if method == "lorma" else base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        for p in base.parameters(): p.requires_grad_(False)

    def forward(self, x):
        if self.method == "lorma":
            y = F.linear(x, self.base.weight, None)
            y = y + self.scale * F.linear(F.linear(y, self.A), self.B)
            return y if self.base.bias is None else y + self.base.bias
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

NF4_VALUES = [-1., -.6961928009986877, -.5250730514526367, -.39491748809814453,
              -.28444138169288635, -.18477343022823334, -.09105003625154495, 0.,
              .07958029955625534, .16093020141124725, .24611230194568634,
              .33791524171829224, .44070982933044434, .5626170039176941,
              .7229568362236023, 1.]

class NF4Linear(nn.Module):
    """Packed frozen NF4 base. Dense dequantization per call; not a fused CUDA kernel."""
    def __init__(self, base, block=64):
        super().__init__()
        self.in_features, self.out_features = base.in_features, base.out_features
        self.shape, self.count, self.block = tuple(base.weight.shape), base.weight.numel(), block
        table = torch.tensor(NF4_VALUES, dtype=torch.float32)
        flat = base.weight.detach().float().cpu().flatten()
        flat = F.pad(flat, (0, (-flat.numel()) % block)).view(-1, block)
        scales = flat.abs().amax(1).clamp_min(1e-12)
        codes = []
        # Bound temporary [blocks, blocksize, 16] memory while quantizing on CPU.
        for start in range(0, len(flat), 1024):
            norm = flat[start:start+1024] / scales[start:start+1024, None]
            codes.append((norm[..., None] - table).abs().argmin(-1).flatten().to(torch.uint8))
        codes = torch.cat(codes)
        self.register_buffer("packed", codes[::2] | (codes[1::2] << 4))
        self.register_buffer("scales", scales)
        self.register_buffer("table", table)
        self.register_buffer("bias", None if base.bias is None else base.bias.detach().clone())

    def dequantize(self, dtype):
        ids = torch.stack([self.packed & 15, self.packed >> 4], -1).flatten().long()
        weight = (self.table[ids].view(-1, self.block) * self.scales[:, None]).flatten()[:self.count]
        return weight.reshape(self.shape).to(dtype)

    def forward(self, x):
        dtype = torch.float16 if x.is_cuda and torch.is_autocast_enabled() else x.dtype
        return F.linear(x.to(dtype), self.dequantize(dtype), None if self.bias is None else self.bias.to(dtype))

class Bottleneck(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.down, self.up = nn.Linear(dim, hidden), nn.Linear(hidden, dim)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x): return self.up(F.gelu(self.down(x)))

class PostBlockAdapter(nn.Module):
    """Pfeiffer-style post-block residual; hybrid extension, not a paper-exact Mamba recipe."""
    def __init__(self, base, dim, hidden):
        super().__init__(); self.base = base; self.adapter = Bottleneck(dim, hidden)
    def forward(self, *args, **kwargs):
        result = self.base(*args, **kwargs)
        if isinstance(result, tuple): return (result[0] + self.adapter(result[0]), *result[1:])
        return result + self.adapter(result)
    def step(self, *args, **kwargs):
        x, cache = self.base.step(*args, **kwargs)
        return x + self.adapter(x), cache
    @property
    def cross(self): return self.base.cross
    @property
    def kind(self): return self.base.kind

class HoulsbyBlock(nn.Module):
    """Encoder/decoder adapters after attention residuals and after the FFN residual."""
    def __init__(self, base, dim, hidden, decoder=False):
        super().__init__(); self.base, self.decoder = base, decoder
        self.adapters = nn.ModuleList([Bottleneck(dim, hidden) for _ in range(3 if decoder else 2)])
    @property
    def cross(self): return self.base.cross
    def forward(self, x, *args, **kwargs):
        b = self.base
        if not self.decoder:
            cos, sin = args[:2]; mask = args[2] if len(args)>2 else kwargs.get("mask")
            h, _ = b.attn(b.n1(x), cos=cos, sin=sin, mask=mask)
            x = x + h; x = x + self.adapters[0](x)
            x = x + b.mlp(b.n2(x)); return x + self.adapters[1](x)
        enc_out, enc_mask, cos, sin = args[:4]
        cache = kwargs.get("cache", args[4] if len(args)>4 else None)
        cross_kv = kwargs.get("cross_kv", args[5] if len(args)>5 else None)
        h, cache = b.self_attn(b.n1(x), cos=cos, sin=sin, causal=True, cache=cache)
        x = x + h; x = x + self.adapters[0](x)
        h, _ = b.cross(b.n2(x), ctx=enc_out, mask=enc_mask, kv=cross_kv)
        x = x + h; x = x + self.adapters[1](x)
        x = x + b.mlp(b.n3(x)); return x + self.adapters[2](x), cache

class EncoderPrefixAttention(nn.Module):
    """Learned per-layer encoder K/V prefix. Full encoder-decoder remains present."""
    def __init__(self, base, length):
        super().__init__(); self.base, self.length = base, length
        self.pk = nn.Parameter(torch.randn(1, base.nh, length, base.hd) * .02)
        self.pv = nn.Parameter(torch.randn(1, base.nh, length, base.hd) * .02)
    def forward(self, x, ctx=None, cos=None, sin=None, mask=None, **kwargs):
        b = self.base; batch = x.size(0)
        q, k, v = b._split(b.q(x)), b._split(b.k(x)), b._split(b.v(x))
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        k = torch.cat([self.pk.expand(batch,-1,-1,-1).to(k.dtype), k], 2)
        v = torch.cat([self.pv.expand(batch,-1,-1,-1).to(v.dtype), v], 2)
        if mask is not None: mask = F.pad(mask, (self.length,0), value=True)
        y = F.scaled_dot_product_attention(q,k,v,attn_mask=mask)
        return b.o(y.transpose(1,2).reshape(batch,x.size(1),-1)), None

def replace_linears(module, method):
    count = 0
    for name, child in list(module.named_children()):
        if name == "head": continue  # Preserve shared token embedding/output weights.
        if isinstance(child, nn.Linear):
            base = NF4Linear(child) if method == "qlora" else child
            setattr(module, name, LowRankLinear(base, "lora" if method == "qlora" else method, RANK, ALPHA))
            count += 1
        else: count += replace_linears(child, method)
    return count

def half_depth(model):
    kept = {}
    if KIND == "encdec":
        for attr, cfg_attr in [("enc_blocks","n_enc"),("dec_blocks","n_dec")]:
            blocks = getattr(model,attr); ix = list(range(1,len(blocks),2))
            if not ix: raise ValueError("KD requires >=2 layers in each stack.")
            setattr(model,attr,nn.ModuleList([blocks[i] for i in ix]))
            setattr(model.c,cfg_attr,len(ix)); kept[attr]=ix
    else:
        ix = list(range(1,len(model.blocks),2))
        if not ix: raise ValueError("KD requires >=2 layers.")
        model.blocks = nn.ModuleList([model.blocks[i] for i in ix]); kept["blocks"] = ix
        if KIND == "mamba": model.c.pattern = "".join(model.c.pattern[i] for i in ix)
        else: model.c.n_layer = len(ix)
    return kept

class Summarizer(nn.Module):
    def __init__(self, backbone, method, student=False):
        super().__init__(); self.backbone, self.method = backbone, method
        # Config is mutable during pruning, so do not mutate the global checkpoint config.
        self.backbone.c = copy.deepcopy(backbone.c)
        if KIND == "encdec": self.backbone.grad_ckpt = GRAD_CKPT
        self.kept = half_depth(backbone) if student else {}
        dim = backbone.c.n_embd if KIND == "decoder" else backbone.c.d_model
        for p in backbone.parameters(): p.requires_grad_(method in {"full","kd"})
        self.soft_prompt = None
        if method in {"lora","lorma","qlora"}:
            print("Adapted linear layers:", replace_linears(backbone,method))
        elif method == "prompt":
            self.soft_prompt = nn.Parameter(torch.randn(1,PROMPT_TOKENS,dim)*.02)
        elif method == "prefix":
            if KIND != "encdec": raise ValueError("Use encoder-decoder for the prefix preset.")
            for b in backbone.enc_blocks: b.attn = EncoderPrefixAttention(b.attn,PROMPT_TOKENS)
        elif method == "adapter":
            attrs = ["enc_blocks","dec_blocks"] if KIND == "encdec" else ["blocks"]
            for attr in attrs:
                setattr(backbone,attr,nn.ModuleList([PostBlockAdapter(b,dim,ADAPTER_DIM) for b in getattr(backbone,attr)]))
        elif method == "houlsby":
            if KIND != "encdec": raise ValueError("Houlsby preset is defined for encoder-decoder.")
            backbone.enc_blocks = nn.ModuleList([HoulsbyBlock(b,dim,ADAPTER_DIM) for b in backbone.enc_blocks])
            backbone.dec_blocks = nn.ModuleList([HoulsbyBlock(b,dim,ADAPTER_DIM,True) for b in backbone.dec_blocks])
        elif method not in {"full","kd"}: raise ValueError(method)

    def encode(self, ids):
        return self.backbone.encode(ids)  # full encoder; source is unpadded micro-batch size 1

    def causal(self, ids, cache=False, last_only=False):
        m = self.backbone; x = m.tok_emb(ids)
        if self.soft_prompt is not None: x = torch.cat([self.soft_prompt.expand(x.size(0),-1,-1),x],1)
        length = x.size(1)
        if length > m.c.n_ctx: raise ValueError("Causal context exceeded.")
        cos,sin = m.cos[:length][None,None],m.sin[:length][None,None]
        # An explicit causal mask is essential when initializing an attention KV cache.
        mask = torch.ones(length,length,dtype=torch.bool,device=x.device).tril()[None,None] if cache else None
        caches=[]
        for block in m.blocks:
            def run(h, b=block):
                if KIND == "mamba": return b(h,cos,sin,mask,None,cache)
                return b(h,cos,sin,mask,(None,None) if cache else None)
            if GRAD_CKPT and self.training and not cache:
                x=torch.utils.checkpoint.checkpoint(lambda h,b=block: (
                    b(h,cos,sin,None,None,False)[0] if KIND=="mamba" else b(h,cos,sin)[0]),x,use_reentrant=False)
            else:
                x, state=run(x)
                if cache: caches.append(state)
        x=m.norm(x)
        if last_only: x=x[:,-1:]
        elif self.soft_prompt is not None: x=x[:,PROMPT_TOKENS:]
        return m.head(x),caches,length

    def forward(self, example):
        if KIND == "encdec":
            return self.backbone(example["source"],example["input"])[0]
        return self.causal(example["input"])[0]

def trainable_parameters(model): return [p for p in model.parameters() if p.requires_grad]



## 5. Phase-6 data, held-out split and long documents

Uses the same HF repository/Kaggle secret as Phase 5, or local JSON paths. There is no public
dev split. Records sharing an ID or normalized document are grouped before splitting, and
exact document-summary duplicates are removed. Identical repeated test IDs collapse to one
record; conflicting duplicate IDs fail. Original string/integer ID types are preserved.

Document token arrays are cached to disk, using the tokenizer hash and document hash.
Uniform ordered windows or head/tail selection share a single source-token budget. This is
lossy extractive context selection, not hierarchical full-document summarization; selected
token coverage is recorded. No pretrained selector, external legal corpus or external LLM
is used. Arbitrary token cuts can split sentences, so inspect coverage and generated examples.

Training uses one unpadded document per microbatch, gradient accumulation and target-only CE.
Causal labels are shifted once; the first summary token and EOS are supervised. No prompt
loss is included. Seq2seq targets are shifted with configurable decoder-start ID; the reference
does not expose the Phase-1 BOS convention, so default EOS-as-start is **unverified**.

Training rows whose summaries are missing, non-string, or empty after Phase-1 normalization are skipped **before** the train/dev split. Skipped IDs/reasons are saved to `data_audit_train.json`. Test document and ID validation remains strict; test rows are never removed for missing summaries.


In [31]:
def id_key(value):
    if isinstance(value,bool) or not isinstance(value,(str,int)):
        raise ValueError(f"ID must be a string or integer, got {type(value).__name__}")
    return (type(value).__name__,value)

def read_records(path, labelled):
    obj=json.loads(Path(path).read_text(encoding="utf-8"))
    if isinstance(obj,dict) and len(obj)==1 and isinstance(next(iter(obj.values())),list): obj=next(iter(obj.values()))
    if not isinstance(obj,list): raise ValueError(f"Expected a JSON list: {path}")
    records=[]; seen={}; skipped=[]
    for row_index,row in enumerate(obj):
        required={"id","document"}
        if not isinstance(row,dict) or not required<=row.keys(): raise ValueError(f"Missing keys: {row!r}")
        key=id_key(row["id"])
        if not isinstance(row["document"],str) or not row["document"].strip(): raise ValueError(f"Empty document: {key}")
        if labelled:
            summary=row.get("summary")
            reason=("missing_summary" if "summary" not in row else
                    "non_string_summary" if not isinstance(summary,str) else
                    "empty_summary_after_normalization" if not normalize_text(summary,False).strip() else None)
            if reason is not None:
                skipped.append({"row_index":row_index,"id":row["id"],"reason":reason,
                                "value_type":type(summary).__name__})
                continue  # No usable supervised target. Never invent one or train on an empty target.
            required=required|{"summary"}
        clean={k:row[k] for k in required}
        if not labelled and key in seen:
            if seen[key]!=clean["document"]: raise ValueError(f"Conflicting test ID: {key}")
            continue  # identical repeated test ID -> one prediction
        seen[key]=clean["document"]
        clean["doc_hash"]=hashlib.sha256(normalize_text(clean["document"],False).encode()).hexdigest()
        records.append(clean)
    print(Path(path).name,":",len(obj),"raw ->",len(records),"records")
    if labelled:
        audit={"source":str(path),"raw_records":len(obj),"retained_records":len(records),
               "skipped_summary_records":len(skipped),"skipped":skipped}
        audit_path=EXP_DIR/("data_audit_"+Path(path).stem+".json")
        audit_path.write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding="utf-8")
        if skipped:
            print(f"Skipped {len(skipped)} training rows without usable string summaries. "
                  f"First IDs: {[r['id'] for r in skipped[:10]]}. Audit: {audit_path}")
            if any(r["reason"]=="non_string_summary" for r in skipped):
                print("Non-string summary values were not coerced to text; inspect audit value_type if your dataset uses structured summaries.")
    if not records:
        raise ValueError("No usable labelled records remain; inspect the summary schema and data audit." if labelled else "Test dataset is empty.")
    return records

def grouped_split(records):
    # Union records sharing either an ID OR a normalized document: neither may leak across split.
    parent=list(range(len(records)))
    def root(i):
        while parent[i]!=i: parent[i]=parent[parent[i]]; i=parent[i]
        return i
    seen={}; unique=[]; exact=set()
    for i,r in enumerate(records):
        pair=(r["doc_hash"],normalize_text(r["summary"],False))
        for key in [("id",id_key(r["id"])),("doc",r["doc_hash"])]:
            if key in seen: parent[root(i)]=root(seen[key])
            else: seen[key]=i
        if pair not in exact: exact.add(pair); unique.append(i)
    groups=defaultdict(list)
    for i in unique: groups[root(i)].append(records[i])
    keys=sorted(groups); random.Random(SEED).shuffle(keys)
    if len(keys)<2: raise ValueError("Need >=2 independent document/ID groups for train/dev.")
    count=max(1,min(len(keys)-1,round(len(keys)*DEV_FRACTION)))
    dev=[r for key in keys[:count] for r in groups[key]]
    train=[r for key in keys[count:] for r in groups[key]]
    assert not {r["doc_hash"] for r in train}&{r["doc_hash"] for r in dev}
    assert not {id_key(r["id"]) for r in train}&{id_key(r["id"]) for r in dev}
    return train,dev

def load_data():
    if LOCAL_TRAIN_JSON and LOCAL_TEST_JSON: paths=[LOCAL_TRAIN_JSON,LOCAL_TEST_JSON]
    else:
        from huggingface_hub import HfApi, hf_hub_download
        token=os.environ.get("HF_TOKEN")
        if not token:
            try:
                from kaggle_secrets import UserSecretsClient
                token=UserSecretsClient().get_secret("capstone-phase1-token")
            except Exception: pass
        files=HfApi().list_repo_files(HF_DATASET_REPO,repo_type="dataset",token=token,revision=HF_REVISION)
        paths=[]
        for filename in ["train.json","test.json"]:
            hits=[f for f in files if "phase_6_summ/" in f.lower() and f.split("/")[-1].lower()==filename]
            if len(hits)!=1: raise ValueError(f"Expected one Phase_6_Summ/{filename}, found {hits}")
            paths.append(hf_hub_download(HF_DATASET_REPO,hits[0],repo_type="dataset",token=token,revision=HF_REVISION))
    train,dev=grouped_split(read_records(paths[0],True)); test=read_records(paths[1],False)
    test_hashes={r["doc_hash"] for r in test}
    print("Train/dev/test:",len(train),len(dev),len(test),"| public-test document overlaps:",
          sum(r["doc_hash"] in test_hashes for r in train+dev))
    # Do not use test labels or derive fitted selectors from test data.
    return train,dev,test,paths

def selected_source(record, limit):
    path=TOKEN_CACHE_DIR/(record["doc_hash"]+".npy")
    if path.is_file(): ids=np.load(path,mmap_mode="r")
    else:
        ids=np.asarray(tokenizer.encode(normalize_text(record["document"],False)),dtype=np.int32)
        np.save(path,ids)
    n=len(ids)
    if n<=limit: out=list(map(int,ids))
    elif CONTEXT_POLICY=="head_tail":
        head=(limit+1)//2; out=list(map(int,ids[:head]))+list(map(int,ids[-(limit-head):]))
    elif CONTEXT_POLICY=="uniform":
        windows=min(SELECTION_WINDOWS,limit)
        widths=[limit//windows+(i<limit%windows) for i in range(windows)]
        # One window per disjoint document region, preserving document order.
        out=[]
        for i,width in enumerate(widths):
            left=n*i//windows; right=n*(i+1)//windows
            start=left if i==0 else right-width if i==windows-1 else (left+right-width)//2
            out.extend(map(int,ids[start:start+width]))
    else: raise ValueError("CONTEXT_POLICY must be uniform or head_tail.")
    return out,{"document_tokens":n,"selected_tokens":len(out),"coverage":min(1.,limit/max(n,1))}

def prepare_example(record, method, labelled=True, device=None):
    device=DEVICE if device is None else device
    virtual=PROMPT_TOKENS if method=="prompt" else 0
    prefix=tokenizer.encode("Document: "); suffix=tokenizer.encode(" Summary:")
    limit=min(MAX_SOURCE_TOKENS,p1_cfg.n_ctx-1) if KIND=="encdec" else min(
        MAX_SOURCE_TOKENS,p1_cfg.n_ctx-MAX_NEW_TOKENS-virtual-len(prefix)-len(suffix))
    if limit<=0: raise ValueError("No context available for document; reduce generation/prompt length.")
    source,coverage=selected_source(record,limit)
    target=None
    if labelled:
        target=tokenizer.encode(normalize_text(record["summary"],False))[:MAX_NEW_TOKENS-1]+[EOS_ID]
    tensor=lambda ids: torch.tensor([ids],dtype=torch.long,device=device)
    if KIND=="encdec":
        if MAX_NEW_TOKENS>p1_cfg.n_ctx: raise ValueError("Target context too long.")
        ex={"source":tensor(source+[EOS_ID]),"coverage":coverage}
        if labelled: ex.update(input=tensor([BOS_ID]+target[:-1]),labels=tensor(target))
    else:
        prompt=prefix+source+suffix
        ex={"prompt":tensor(prompt),"coverage":coverage}
        if labelled:
            ids=prompt+target; labels=ids[1:]
            labels[:len(prompt)-1]=[-100]*(len(prompt)-1)
            ex.update(input=tensor(ids[:-1]),labels=tensor(labels))
    return ex



## 6. Cached generation and local evaluation

Decoder attention: prefill with an explicit causal mask, then one-token KV updates. Hybrid:
parallel prefill, then `step()` with fixed Mamba states and growing attention KV caches.
The entire hybrid is therefore **not constant-memory**. Encoder-decoder: encode once,
project cross-KV once, then decoder self-KV updates. Generation is one document at a time.

Includes repetition penalty, no-repeat n-grams, greedy or temperature/top-k/top-p sampling,
minimum-length EOS suppression, maximum length and RoPE-context enforcement. The n-gram
constraint is relaxed only if it removes every candidate. Non-text extra vocab rows are banned.
Empty decoded summaries fail explicitly, rather than becoming copied-document fallbacks.

Primary BERTScore is computed by the competition server. No external pretrained scoring
weights are downloaded. Local ROUGE uses lowercase alphanumeric tokens without stemming;
it is an approximation of the unprovided official scorer. Best checkpoint uses local ROUGE-L.


In [32]:
def sync():
    if DEVICE.type=="cuda": torch.cuda.synchronize()

def amp(): return torch.autocast("cuda",dtype=torch.float16) if DEVICE.type=="cuda" else nullcontext()

def controlled_token(logits, generated, force_length=False):
    logits=logits.float().clone().flatten()
    if not torch.isfinite(logits).all():
        raise FloatingPointError("Nonfinite generation logits; inspect FP16 stability before submitting.")
    logits[tokenizer.vocab_size:]=-float("inf")  # no sentinel/padded-vocabulary output
    if generated and REPETITION_PENALTY!=1:
        ix=torch.tensor(sorted(set(generated)),device=logits.device)
        values=logits[ix]; logits[ix]=torch.where(values<0,values*REPETITION_PENALTY,values/REPETITION_PENALTY)
    if len(generated)<MIN_NEW_TOKENS or force_length: logits[EOS_ID]=-float("inf")
    # A byte-level BPE can otherwise generate only whitespace for a nonempty token sequence.
    if len(generated)>=MIN_NEW_TOKENS and not tokenizer.decode(generated).strip(): logits[EOS_ID]=-float("inf")
    before_ngram=logits.clone()
    n=NO_REPEAT_NGRAM
    if n>0 and len(generated)>=n-1:
        prefix=tuple(generated[-(n-1):]) if n>1 else ()
        banned={generated[j+n-1] for j in range(len(generated)-n+1) if tuple(generated[j:j+n-1])==prefix}
        if banned: logits[list(banned)]=-float("inf")
    if not torch.isfinite(logits).any(): logits=before_ngram
    if not torch.isfinite(logits).any(): raise FloatingPointError("No finite generation candidates.")
    if not DO_SAMPLE: return int(logits.argmax())
    logits=logits/TEMPERATURE
    if TOP_K>0:
        k=min(TOP_K,logits.numel()); logits[logits<torch.topk(logits,k).values[-1]]=-float("inf")
    if TOP_P<1:
        values,order=logits.sort(descending=True)
        remove=values.softmax(-1).cumsum(-1)>TOP_P
        remove[1:]=remove[:-1].clone(); remove[0]=False
        logits[order[remove]]=-float("inf")
    return int(torch.multinomial(logits.softmax(-1),1))

@torch.inference_mode()
def generate_ids(model, record, force_length=False, deadline=None):
    model.eval(); ex=prepare_example(record,model.method,False)
    m=model.backbone; result=[]
    with amp():
        if KIND=="encdec":
            enc=model.encode(ex["source"]); cross=m.cross_kv(enc)
            caches=[(None,None) for _ in m.dec_blocks]
            logits,caches=m.decode_step(torch.tensor([[BOS_ID]],device=DEVICE),0,caches,cross,None)
            next_pos=1
        else:
            logits,caches,next_pos=model.causal(ex["prompt"],True,True)
        for step in range(MAX_NEW_TOKENS):
            if deadline is not None and time.monotonic()>deadline: raise TimeoutError("Generation time budget exhausted; partial predictions saved, no incomplete ZIP emitted.")
            token=controlled_token(logits[0,-1],result,force_length)
            if token==EOS_ID: break
            result.append(token)
            if step+1==MAX_NEW_TOKENS: break
            tok=torch.tensor([[token]],device=DEVICE)
            if KIND=="encdec": logits,caches=m.decode_step(tok,next_pos,caches,cross,None)
            elif KIND=="mamba": logits,caches=m.step(tok,torch.tensor([next_pos],device=DEVICE),None,caches)
            else:
                logits,_,caches=m(tok,pos_ids=torch.tensor([[next_pos]],device=DEVICE),caches=caches,last_only=True)
            next_pos+=1
    return result,ex["coverage"]

def generate_text(model, record, **kwargs):
    ids,coverage=generate_ids(model,record,**kwargs)
    text=normalize_text(tokenizer.decode(ids),False)
    if not text: raise ValueError(f"Model generated an empty summary for {record['id']!r}; refusing a dummy submission.")
    return text,coverage

def rouge_f1(prediction,reference):
    # Transparent local metric: lowercased alphanumeric tokens, no stemming.
    p=re.findall(r"[a-z0-9]+",prediction.lower()); r=re.findall(r"[a-z0-9]+",reference.lower())
    overlap=sum((Counter(p)&Counter(r)).values())
    f=lambda hit: 2*hit/(len(p)+len(r)) if p or r else 0.
    row=[0]*(len(r)+1)
    for a in p:
        prev=0
        for j,b in enumerate(r,1):
            old=row[j]; row[j]=prev+1 if a==b else max(row[j],row[j-1]); prev=old
    return {"rouge1_f1":f(overlap),"rougeL_f1":f(row[-1])}



## 7. Throughput benchmark, budget auto-fit, training and KD

Benchmark runs genuine gradients/optimizer steps with LR=0, then discards optimizer state.
Generation benchmarks force the max output length to avoid optimistic early-EOS timing.
Measured training samples cover document-length quantiles; a configurable margin is applied.
Actual load can still vary. Token cache creation is lazy; benchmark measurements include
tokenization when not cached. Time limits are checked between updates/documents/tokens.

Full teacher is first fine-tuned on train-only summaries within its reserved budget, saved,
then frozen. The half-depth student learns aligned same-tokenizer teacher logits online:
`(1-alpha)*CE + alpha*T^2*KL(teacher || student)` on summary/EOS positions only. Teacher
confidence/quality is unverified; teacher pretraining alone is not claimed to teach summarization.
Validation/test references never train the teacher or student. No cross-tokenizer logit KD.

CUDA uses float32 parameters, FP16 autocast and GradScaler; clipping follows unscaling.
CPU tests use FP32. Nonfinite losses/gradients fail explicitly. One document per microbatch
also avoids the reference hybrid's padding-mask/checkpointing hazard. Checkpoint selection
uses a seeded fixed dev subset (default 32); record this when comparing experiments.


In [33]:
def supervised_loss(model,record,teacher=None):
    ex=prepare_example(record,model.method)
    with amp(): logits=model(ex)
    mask=ex["labels"]!=-100; labels=ex["labels"][mask]
    z=logits[mask].float()
    ce=F.cross_entropy(z,labels)
    if teacher is None: return ce
    with torch.no_grad(),amp(): tz=teacher(ex)[mask].float()
    temperature=KD_TEMPERATURE
    kl=F.kl_div(F.log_softmax(z/temperature,-1),F.softmax(tz/temperature,-1),reduction="batchmean")
    return (1-KD_ALPHA)*ce+KD_ALPHA*temperature**2*kl

def optimizer_for(model,lr): return torch.optim.AdamW(trainable_parameters(model),lr=lr,weight_decay=WEIGHT_DECAY)

def update_batch(model,records,opt,scaler,teacher=None):
    model.train(); opt.zero_grad(set_to_none=True); loss_sum=0.
    for record in records:
        loss=supervised_loss(model,record,teacher)
        if not torch.isfinite(loss): raise FloatingPointError("Nonfinite training loss; reduce LR/length and inspect FP16 stability.")
        scaler.scale(loss/len(records)).backward(); loss_sum+=float(loss.detach())
    scaler.unscale_(opt)
    norm=torch.nn.utils.clip_grad_norm_(trainable_parameters(model),CLIP)
    if not torch.isfinite(norm):
        scaler.update(); opt.zero_grad(set_to_none=True)
        raise FloatingPointError("Nonfinite gradient norm; no checkpoint/submission accepted.")
    scaler.step(opt); scaler.update()
    return loss_sum/len(records)

def benchmark(model,train,dev,test,teacher=None):
    # LR=0: run real backward + optimizer work without changing the checkpoint weights.
    opt=optimizer_for(model,0.); scaler=torch.amp.GradScaler("cuda",enabled=DEVICE.type=="cuda")
    ordered=sorted(train,key=lambda r:len(r["document"]))
    probe=[ordered[int(q*(len(ordered)-1))] for q in [.5,.75,.9,1.]]
    update_batch(model,probe[:1],opt,scaler,teacher); sync()
    times=[]
    for record in probe:
        sync(); start=time.monotonic(); update_batch(model,[record],opt,scaler,teacher); sync()
        times.append(time.monotonic()-start)
    opt.zero_grad(set_to_none=True); del opt,scaler
    # Force max-length generation so an untrained model's premature EOS cannot underbudget.
    gpool=sorted(dev+test,key=lambda r:len(r["document"]))
    gen_times=[]
    for q in [.5,1.]:
        record=gpool[int(q*(len(gpool)-1))]; sync(); start=time.monotonic()
        generate_ids(model,record,force_length=True); sync(); gen_times.append(time.monotonic()-start)
    train_sec=max(times)*BENCHMARK_MARGIN
    gen_sec=max(gen_times)*BENCHMARK_MARGIN
    print(f"Benchmark: {train_sec:.3f} s/train document; {gen_sec:.3f} s/max-length generated document (margin included)")
    return {"train_seconds_per_document":train_sec,"generation_seconds_per_document":gen_sec,
            "raw_train_seconds":times,"raw_generation_seconds":gen_times}

def cpu_state(model): return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def atomic_save(payload,path):
    path=Path(path); tmp=path.with_suffix(path.suffix+".tmp"); torch.save(payload,tmp); os.replace(tmp,path)

def save_model(model,path,metadata):
    atomic_save({"model":cpu_state(model),"phase1_cfg":ck["cfg"],"phase1_path":str(CKPT_PATH),
                 "kind":KIND,"method":model.method,"kept_layers":model.kept,
                 "tokenizer_sha256":TOKENIZER_SHA256,"experiment":EXPERIMENT,
                 "settings":run_settings(),"metadata":metadata},path)

def fit_teacher(teacher,train,deadline):
    # Full-depth summarization SFT, train only. No validation/test targets are used.
    opt=optimizer_for(teacher,TEACHER_LR); scaler=torch.amp.GradScaler("cuda",enabled=DEVICE.type=="cuda")
    rng=random.Random(SEED+100); updates=0; losses=[]
    while time.monotonic()<deadline and updates<TEACHER_MAX_UPDATES:
        batch=[train[rng.randrange(len(train))] for _ in range(GRAD_ACCUM)]
        losses.append(update_batch(teacher,batch,opt,scaler)); updates+=1
        if updates%20==0: print("Teacher updates:",updates,"loss:",round(float(np.mean(losses[-20:])),4))
    if updates==0: raise RuntimeError("No budget for teacher SFT; increase total budget.")
    opt.zero_grad(set_to_none=True); del opt,scaler
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad_(False)
    save_model(teacher,EXP_DIR/f"teacher_{EXPERIMENT}.pt",{"updates":updates,"mean_loss":float(np.mean(losses))})
    return {"updates":updates,"mean_loss":float(np.mean(losses))}

@torch.inference_mode()
def evaluate(model,dev,deadline):
    model.eval(); total={"rouge1_f1":0.,"rougeL_f1":0.}; examples=[]
    for row in dev:
        text,coverage=generate_text(model,row,deadline=deadline)
        scores=rouge_f1(text,row["summary"])
        for key in total: total[key]+=scores[key]
        examples.append({"id":row["id"],"prediction":text,"reference":row["summary"],**scores,**coverage})
    return {key:value/len(dev) for key,value in total.items()},examples

def fit_student(model,train,dev,teacher,bench,deadline,remaining_seconds):
    reserve_infer=bench["generation_seconds_per_document"]*len(TEST_RECORDS)
    reserve_dev=bench["generation_seconds_per_document"]*len(dev)*EPOCHS
    available=remaining_seconds-reserve_infer-reserve_dev-CHECKPOINT_RESERVE_SECONDS
    docs=min(len(train),int(max(0,available)/(EPOCHS*bench["train_seconds_per_document"])))
    if TRAIN_DOCS_PER_EPOCH is not None: docs=min(docs,TRAIN_DOCS_PER_EPOCH)
    if docs<1:
        raise RuntimeError(f"Budget cannot fit validation+all test predictions (estimated {reserve_infer+reserve_dev:.0f}s) "
                           "and training. Increase TOTAL_MINUTES or reduce MAX_NEW_TOKENS/MAX_DEV_DOCS; never drop test rows.")
    print(f"AUTO-FIT: {docs} documents/epoch x {EPOCHS} epochs; test reserve {reserve_infer/60:.1f} min")
    opt=optimizer_for(model,EXP["lr"]); scaler=torch.amp.GradScaler("cuda",enabled=DEVICE.type=="cuda")
    steps=max(1,EPOCHS*math.ceil(docs/GRAD_ACCUM)); warmup=max(1,round(steps*WARMUP_FRACTION))
    done=0; best=-float("inf"); history=[]; rng=random.Random(SEED); best_path=EXP_DIR/f"best_{EXPERIMENT}.pt"
    best_written=False  # Never accidentally restore a prior run's checkpoint after a timeout.
    for epoch in range(EPOCHS):
        sample=rng.sample(train,docs); losses=[]
        for start in range(0,len(sample),GRAD_ACCUM):
            # Leave time for this epoch's dev pass, final test inference and checkpoint I/O.
            if time.monotonic()+reserve_infer+bench["generation_seconds_per_document"]*len(dev)+CHECKPOINT_RESERVE_SECONDS>=deadline: break
            scale=(done+1)/warmup if done<warmup else .5*(1+math.cos(math.pi*(done-warmup)/max(1,steps-warmup)))
            for group in opt.param_groups: group["lr"]=EXP["lr"]*max(.05,scale)
            losses.append(update_batch(model,sample[start:start+GRAD_ACCUM],opt,scaler,teacher)); done+=1
            if done%20==0: print(f"epoch {epoch+1} update {done}/{steps}, loss {np.mean(losses[-20:]):.4f}")
        if not losses:
            if not best_written: raise TimeoutError("No completed student update within budget.")
            break
        metrics,examples=evaluate(model,dev,deadline-reserve_infer-CHECKPOINT_RESERVE_SECONDS)
        row={"epoch":epoch+1,"train_loss":float(np.mean(losses)),"updates":len(losses),**metrics}; history.append(row); print(row)
        if metrics["rougeL_f1"]>best:
            best=metrics["rougeL_f1"]; save_model(model,best_path,row)
            best_written=True
            (EXP_DIR/f"dev_examples_{EXPERIMENT}.json").write_text(json.dumps(examples,ensure_ascii=False,indent=2))
        if len(losses)<math.ceil(docs/GRAD_ACCUM): break
    opt.zero_grad(set_to_none=True); del opt,scaler
    saved=torch.load(best_path,map_location="cpu",weights_only=False)
    model.load_state_dict(saved["model"],strict=True); del saved
    return history,docs,best_path



## 8. Strict submission and results

The output is exactly a JSON list of `{id, summary}` records in unique test-ID order.
Validation checks key sets, ID types, coverage, uniqueness and nonempty strings. The ZIP
must contain exactly `prediction.json` at root. Each model/preset has its own directory.
Best checkpoint, teacher checkpoint (KD), split manifest, dev examples, partial prediction
JSONL and results JSON are also retained. A timeout leaves partial predictions but no new
incomplete submission ZIP. Inspect failures before reusing older output files from a run.


In [34]:
def export_submission(predictions,test):
    expected=[id_key(r["id"]) for r in test]
    got=[]
    for row in predictions:
        if set(row)!={"id","summary"}: raise ValueError("Output keys must be exactly id and summary.")
        if not isinstance(row["summary"],str) or not row["summary"].strip(): raise ValueError("Empty summary.")
        got.append(id_key(row["id"]))
    if got!=expected or len(got)!=len(set(got)): raise ValueError("Test ID coverage/order/uniqueness mismatch.")
    path=EXP_DIR/"prediction.json"
    path.write_text(json.dumps(predictions,ensure_ascii=False,allow_nan=False,indent=2),encoding="utf-8")
    archive=EXP_DIR/f"submission_{EXPERIMENT}.zip"
    with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as z: z.write(path,arcname="prediction.json")
    with zipfile.ZipFile(archive) as z:
        assert z.namelist()==["prediction.json"]
        assert json.loads(z.read("prediction.json"))==predictions
    print("SUBMISSION:",archive,"| summaries:",len(predictions)); return archive

def run_settings():
    names=["EXPERIMENT","KIND","SEED","EPOCHS","MAX_SOURCE_TOKENS","MAX_NEW_TOKENS","MIN_NEW_TOKENS",
           "CONTEXT_POLICY","SELECTION_WINDOWS","DO_SAMPLE","TEMPERATURE","TOP_P","TOP_K",
           "REPETITION_PENALTY","NO_REPEAT_NGRAM","TOTAL_MINUTES","RANK","ALPHA","ADAPTER_DIM",
           "PROMPT_TOKENS","GRAD_ACCUM","GRAD_CKPT","DEV_FRACTION","MAX_DEV_DOCS","DECODER_START_ID",
           "KD_ALPHA","KD_TEMPERATURE","HF_REVISION"]
    return {name:globals()[name] for name in names}

def main():
    global TOKEN_CACHE_DIR,TEST_RECORDS
    start=time.monotonic(); deadline=start+TOTAL_MINUTES*60
    print("[1/6] Loading your trained Phase-1 checkpoint")
    load_assets()  # ALWAYS loads your Phase-1 checkpoint before constructing any training model.
    TOKEN_CACHE_DIR=EXP_DIR/("tokens_"+TOKENIZER_SHA256[:16]); TOKEN_CACHE_DIR.mkdir(parents=True,exist_ok=True)
    print("[2/6] Loading and auditing Phase-6 data, then creating train/dev split")
    train,dev,TEST_RECORDS,paths=load_data()
    rng=random.Random(SEED+1); rng.shuffle(dev)
    dev_eval=dev[:MAX_DEV_DOCS] if MAX_DEV_DOCS is not None else dev
    (EXP_DIR/"split_manifest.json").write_text(json.dumps({"train_ids":[r["id"] for r in train],
        "dev_ids":[r["id"] for r in dev],"evaluated_dev_ids":[r["id"] for r in dev_eval],"seed":SEED},indent=2))
    teacher=None; teacher_info=None
    if EXP["method"]=="kd":
        teacher=Summarizer(build_backbone(),"full").to(DEVICE)
        student=Summarizer(build_backbone(),"kd",student=True).to(DEVICE)
        tp=sum(p.numel() for p in teacher.parameters()); sp=sum(p.numel() for p in student.parameters())
        assert sp<tp; print("KD parameters teacher/student:",tp,sp,"| kept:",student.kept)
    else: student=Summarizer(build_backbone(),EXP["method"]).to(DEVICE)
    print("Loaded Phase-1 tensors BEFORE adaptation. Trainable parameters:",sum(p.numel() for p in trainable_parameters(student)))
    if teacher is not None:
        teacher.eval()
        for p in teacher.parameters(): p.requires_grad_(False)
    if DEVICE.type=="cuda": torch.cuda.reset_peak_memory_stats()
    print("[3/6] Benchmarking throughput (LR=0; checkpoint weights unchanged)")
    bench=benchmark(student,train,dev_eval,TEST_RECORDS,teacher)
    safety=TOTAL_MINUTES*60*SAFETY_FRACTION
    fixed=bench["generation_seconds_per_document"]*(len(TEST_RECORDS)+len(dev_eval)*EPOCHS)+CHECKPOINT_RESERVE_SECONDS
    available=deadline-time.monotonic()-safety-fixed
    if available<=0: raise RuntimeError("Measured fixed validation/test cost exceeds budget; increase TOTAL_MINUTES or reduce generation length.")
    if teacher is not None:
        print("[4/6] Fine-tuning the full-depth teacher on training summaries")
        teacher_deadline=time.monotonic()+available*TEACHER_BUDGET_FRACTION
        for p in teacher.parameters(): p.requires_grad_(True)
        teacher_info=fit_teacher(teacher,train,teacher_deadline)
    remaining=deadline-time.monotonic()-safety
    print("[4/6] Training the selected model; validating and saving the best checkpoint")
    history,docs,best_path=fit_student(student,train,dev_eval,teacher,bench,deadline-safety,remaining)
    if teacher is not None: del teacher
    if DEVICE.type=="cuda": torch.cuda.empty_cache()
    print("[5/6] Generating summaries for every test ID using the best checkpoint")
    predictions=[]; coverage=[]; progress=EXP_DIR/f"predictions_partial_{EXPERIMENT}.jsonl"
    with progress.open("w",encoding="utf-8") as f:
        for i,row in enumerate(TEST_RECORDS):
            text,stats=generate_text(student,row,deadline=deadline)
            item={"id":row["id"],"summary":text}; predictions.append(item); coverage.append(stats)
            f.write(json.dumps(item,ensure_ascii=False)+"\n"); f.flush()
            if (i+1)%25==0: print("Generated",i+1,"/",len(TEST_RECORDS))
    print("[6/6] Validating prediction.json and creating the submission ZIP")
    archive=export_submission(predictions,TEST_RECORDS)
    results={"settings":run_settings(),"phase1_checkpoint":CKPT_PATH,"phase1_step":ck.get("step"),
             "phase1_cfg":ck["cfg"],"tokenizer_sha256":TOKENIZER_SHA256,"benchmarks":bench,
             "history":history,"train_documents_per_epoch":docs,"teacher":teacher_info,"kept_layers":student.kept,
             "dataset_files":[{"path":str(p),"sha256":hashlib.sha256(Path(p).read_bytes()).hexdigest()} for p in paths],
             "counts":{"train":len(train),"dev":len(dev),"dev_evaluated":len(dev_eval),"test":len(TEST_RECORDS)},
             "best_checkpoint":str(best_path),"submission":str(archive),"elapsed_seconds":time.monotonic()-start,
             "peak_cuda_gib":torch.cuda.max_memory_allocated()/2**30 if DEVICE.type=="cuda" else None,
             "mean_test_token_coverage":float(np.mean([r["coverage"] for r in coverage])),
             "bertscore_f1":None,"local_metric":"ROUGE F1: lowercase alphanumeric, no stemming; not official scorer"}
    (EXP_DIR/f"results_{EXPERIMENT}.json").write_text(json.dumps(results,indent=2,allow_nan=False))
    print("Results:",EXP_DIR/f"results_{EXPERIMENT}.json")
    return results


## 9. Run the entire pipeline

This cell **loads Phase 1, trains and generates**, unless you explicitly set `RUN_PIPELINE=False`.
Do not run three complete models inside one notebook/GPU process. Use the three independent files.

Recommended six-method suite across the notebooks: `D1_lorma`, `D4_prompt`, `E1_houlsby`,
`E2_prefix`, `E4_qlora`, `M3_kd`. Optional full-FT baselines and additional LoRA/adapter runs
are provided. Three initial notebook runs are three experiments, not six. The assignment's
fourth Phase-1 architecture is still unavailable; these three do not resolve that requirement.


In [35]:
if RUN_PIPELINE:
    RESULTS = main()


[1/6] Loading your trained Phase-1 checkpoint
PHASE-1 CHECKPOINT: /kaggle/input/models/karankumar9939/cs-781-project-encoder-decoder-model/pytorch/default/1/encoder_decoder_checpoint/best.pt | step: 1076
Checkpoint config: {'vocab_size': 32128, 'n_ctx': 1280, 'd_model': 768, 'n_head': 12, 'n_enc': 6, 'n_dec': 6, 'n_ffn': 2048, 'rope_base': 10000.0}
Tokenizer: /kaggle/input/models/karankumar9939/cs-781-project-encoder-decoder-model/pytorch/default/1/encoder_decoder_checpoint/bpe_merges.json | SHA-256: 8f2503677817a580790dea20ef24ffa1ec710675c54d2a7eaf9e91e3435a1c67
NOTE: hash recorded, but tokenizer pairing is not independently authenticated.
ASSUMPTION: EOS is the decoder-start token; set DECODER_START_ID if Phase 1 used another ID.
[2/6] Loading and auditing Phase-6 data, then creating train/dev split
train.json : 7030 raw -> 7028 records
Skipped 2 training rows without usable string summaries. First IDs: ['299', '4799']. Audit: /kaggle/working/phase6_summ_encdec/E2_prefix/20261103-10

## 10. Continue inference from a saved best checkpoint

If training finished but prediction timed out, use the cell below **instead of rerunning
Section 9**. Set `RUN_PIPELINE=False` before Run All, retain the same preset/settings and
Phase-1 assets, and point `RESTORE_PATH` to the saved `best_<EXP>.pt`. This reconstructs the
same adaptation and pruned depth, strictly loads saved trained weights, and regenerates test
predictions. It does not resume optimizer training. Increase the inference deadline if needed.


In [36]:
RESTORE_PATH = ""  # e.g. /kaggle/input/your-phase6-output/best_M1_adapter.pt
if RESTORE_PATH:
    load_assets()
    TOKEN_CACHE_DIR=EXP_DIR/("tokens_"+TOKENIZER_SHA256[:16]); TOKEN_CACHE_DIR.mkdir(parents=True,exist_ok=True)
    saved=torch.load(RESTORE_PATH,map_location="cpu",weights_only=False)
    if saved["kind"]!=KIND or saved["experiment"]!=EXPERIMENT or saved["tokenizer_sha256"]!=TOKENIZER_SHA256:
        raise ValueError("Checkpoint architecture/experiment/tokenizer mismatch.")
    if saved["phase1_cfg"]!=ck["cfg"]:
        raise ValueError("Restore with the same Phase-1 config, including RoPE settings.")
    # Require the same train-time architectural adaptation settings; generation settings may change.
    for name in ["RANK","ALPHA","ADAPTER_DIM","PROMPT_TOKENS"]:
        if saved["settings"][name]!=globals()[name]: raise ValueError(f"Restore {name} from checkpoint settings.")
    restored=Summarizer(build_backbone(),EXP["method"],student=EXP["method"]=="kd").to(DEVICE)
    restored.load_state_dict(saved["model"],strict=True); del saved
    _,_,test_records,_=load_data()
    deadline=time.monotonic()+TOTAL_MINUTES*60
    predictions=[]
    for record in test_records:
        summary,_=generate_text(restored,record,deadline=deadline)
        predictions.append({"id":record["id"],"summary":summary})
    export_submission(predictions,test_records)


## Source provenance
```json
{
  "reference": "cs781-phase5-lsi-encdec.ipynb",
  "tokenizer_cell": 4,
  "backbone_cell": 5,
  "tokenizer_sha256": "3be37306135fc902156959dc4c3f3002b276828c94f0ca1159e587461e5173a6",
  "backbone_sha256": "1d77003c1e45371d28499065cceeff212713972e02e2e53a6d78c42c26a68259"
}
```

API references: [PyTorch AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html), [Hugging Face downloads](https://huggingface.co/docs/huggingface_hub/en/guides/download).


## Validation performed before delivery

A synthetic CPU harness using PyTorch 2.15.0+cpu exercised this notebook's definition
cells and pipeline functions. No actual Phase-1 weights or Phase-6 data were available here.

- Presets tested: E1_houlsby, E2_prefix, E3_kd, E4_qlora, E5_full, E6_lorma.
- Cached/full-logit equivalence before and after parameter updates; finite backward gradients;
  trainable parameter updates; forced-length generation and output vocabulary constraints.
- Two end-to-end runs for this architecture: default PEFT and KD. These tested file-based
  Phase-1 loading, teacher SFT where applicable, measured budget fitting, student training,
  dev selection, saved-best restore, prediction and root-only ZIP output.
- Inference-only restore and optional gradient-checkpointing backward path passed.
- LORMA matrix identity, packed NF4 storage, EOS/sampling constraints, nonfinite-logit rejection,
  and malformed-submission rejection passed.
- Tokenizer/model cells match the supplied Phase-5 reference exactly; notebook schema validates.

Across the three notebooks: **17 preset tests and 6 complete tiny pipeline runs passed**.
This does not establish Kaggle T4/FP16 numerical stability, full-scale memory use, 90-minute
feasibility, official score equivalence or summary quality. Inspect the printed Phase-1 path,
config, tokenizer hash, trainable parameters and measured seconds/document on your real run.


## Data-loader correction
The reader now skips and audits unusable labelled summaries, including the reported empty-summary case. It does not fabricate replacement targets or relax test-ID/document validation. Pipeline stage messages identify real training, generation and ZIP export. Targeted synthetic parser regression tests passed; the actual Hugging Face row 299 was not inspected here.
